In [1]:
# import 
import pandas as pd
import string
import re
from src.preprocesing import preprocesing_text
from src.tokenization import tokenizer

In [2]:
# read data
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")
train_df

,comment,label,label_id
0,غذا خیلی سرد بود در صورتیکه فاصله ما خیلی کم است,SAD,1.0
1,بهتره بتونیم ران یا سینه رو خودمون انتخاب کنیم,HAPPY,0.0
2,غذا بد بود حالم خیییییلی بده. دل دردو دل پیچه....,SAD,1.0
3,با سلام سابق بر این بسته بندی از کیفیت بهتری ب...,SAD,1.0
4,سلام، خیلی ممنون و متشکرم,HAPPY,0.0
...,...,...,...
52105,یکی از بهترین ته چین هاییه که خوردم,HAPPY,0.0
52106,به موقع و خوب دستتون درد نکنه,HAPPY,0.0
52107,خیلی تازه و خوشمزه بود. بسته بندی شیک وعالی. خ...,HAPPY,0.0
52108,فوق العاده سریع و عالی واقعا واسه همه زود آورد...,HAPPY,0.0


In [3]:
test_df

,comment,label,label_id
0,استرس داشتم نکنه که به خاطر تخفیف از کیفیت پای...,HAPPY,0.0
1,قاشق و نمک نداشت,HAPPY,0.0
2,سس مخصوص ساده‌ترین سس مایونزی بود که میشد تصور...,SAD,1.0
3,به جای شامپو. نرم کننده اوردن,SAD,1.0
4,یک غذای خوب وسالم ممنون,SAD,1.0
...,...,...,...
9028,بعضی از تیکه‌ها تند بود و در منو قید نشده بود....,SAD,1.0
9029,پیتزا گوشت و قارچ سفارش داده شده و نصف پیتزا ک...,SAD,1.0
9030,عالی بود سریع ویخ زده,HAPPY,0.0
9031,غذای خوبی بود نسبت به قیمتش,HAPPY,0.0


# EDA & preprocesing

In [4]:
train_df["label"].value_counts(), test_df["label"].value_counts()

(label
 HAPPY    26236
 SAD      25874
 Name: count, dtype: int64,
 label
 SAD      4547
 HAPPY    4486
 Name: count, dtype: int64)

In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52110 entries, 0 to 52109
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   comment   52110 non-null  object 
 1   label     52110 non-null  object 
 2   label_id  52110 non-null  float64
dtypes: float64(1), object(2)
memory usage: 1.2+ MB


### check text for cleaning

In [6]:
# check url
train_df[train_df["comment"].str.contains("http", na= False)]

,comment,label,label_id


In [7]:
# check mentions
train_df[train_df["comment"].str.contains("@", na= False)]

,comment,label,label_id
24236,بسیار رستوران بد و غذای سرد و بی کیفیت!!!!! @,SAD,1.0
34930,از زمان سفارش تا رسیدن پیتزا ۲ ساعت و ربع!!!!!...,SAD,1.0
36397,سفارش با یک ساعت تاخیر به دستمان رسید چرا اینق...,SAD,1.0
42184,متاسفانه سفار@ ثبت شده به تعدادرونداشتن تماس گ...,SAD,1.0


In [8]:
# check number
train_df[train_df["comment"].str.contains(r"\d", regex= True, na= False)]

,comment,label,label_id
14,کیفیت پایین‌تر از پایین، در حد یه ساندویچ ۸-۹ ...,SAD,1.0
26,راد عزیز، متأسفانه دقت قبل در ارسال سفارش دیده...,SAD,1.0
29,اصلا فکر نمیکردم به این شدت غذا و کیفیتش پایین...,SAD,1.0
33,کیفیت بسیار بد!!! من ساندویچ ژامبون ویژه سفارش...,SAD,1.0
38,غذا بعد از ۱ ساعت و نیم با رفتار طلبکارانه پیک...,SAD,1.0
...,...,...,...
52088,به معنای واقعی عالی. چقدر تمیز و لایه لایه چید...,HAPPY,0.0
52092,طعم غذا به نسبت خوب بود منتها زمان رسید غذا خی...,HAPPY,0.0
52096,واقعا ۲۱ هزار تومان و مالیات ۲هزار تومنی برای ...,SAD,1.0
52099,سیب زمینی با قارچ و پنیر شامل ۲۰ عدد سیب زمینی...,HAPPY,0.0


In [9]:
# check punctuation
punctuation = string.punctuation + "،؛«»٪" # !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~،؛«»٪
pattern = f"[{re.escape(punctuation)}]" # [!"\#\$%\&'\(\)\*\+,\-\./:;<=>\?@\[\\\]\^_`\{\|\}\~،؛«»٪]
train_df[train_df["comment"].str.contains(pattern, regex= True, na=False)]

,comment,label,label_id
2,غذا بد بود حالم خیییییلی بده. دل دردو دل پیچه....,SAD,1.0
3,با سلام سابق بر این بسته بندی از کیفیت بهتری ب...,SAD,1.0
4,سلام، خیلی ممنون و متشکرم,HAPPY,0.0
6,مواد پیتزا بسیااار کم بود با اینکه قیمت اصلا پ...,HAPPY,0.0
7,کیفیت زیاد خوب نبود،,HAPPY,0.0
...,...,...,...
52096,واقعا ۲۱ هزار تومان و مالیات ۲هزار تومنی برای ...,SAD,1.0
52100,تنها حسنی که داشت زود رسید. تو توضیحات هم نوشت...,HAPPY,0.0
52101,خیلی خوشمزه بودن فقط احساس کردم گوشت چرخ کرده‌...,HAPPY,0.0
52102,یه چیزی که برام جالبه و خیلی هم مهمه.. علاوه ب...,HAPPY,0.0


In [10]:
# apply clean text on dataframe

train_df["clean_text"] = train_df["comment"].apply(preprocesing_text)
test_df["clean_text"] = test_df["comment"].apply(preprocesing_text)


In [11]:
train_df["token"] = train_df["clean_text"].apply(lambda text : tokenizer(text, remove_stop_word=True))
test_df["token"] = test_df["clean_text"].apply(lambda text : tokenizer(text, remove_stop_word=True))

In [12]:
train_df

,comment,label,label_id,clean_text,token
0,غذا خیلی سرد بود در صورتیکه فاصله ما خیلی کم است,SAD,1.0,غذا خیلی سرد بود در صورتیکه فاصله ما خیلی کم است,"[غذا, سرد, صورتیکه, فاصله]"
1,بهتره بتونیم ران یا سینه رو خودمون انتخاب کنیم,HAPPY,0.0,بهتره بتونیم ران یا سینه رو خودمون انتخاب کنیم,"[بهتره, بتونیم, ران, سینه, خودمون, انتخاب]"
2,غذا بد بود حالم خیییییلی بده. دل دردو دل پیچه....,SAD,1.0,غذا بد بود حالم خیلی بده دل دردو دل‌پیچه معلوم...,"[غذا, بد, حالم, بده, دل, دردو, دل‌پیچه, معلوم,..."
3,با سلام سابق بر این بسته بندی از کیفیت بهتری ب...,SAD,1.0,با سلام سابق بر این بسته بندی از کیفیت بهتری ب...,"[سلام, سابق, بسته, کیفیت, بهتری, برخوردار, حاض..."
4,سلام، خیلی ممنون و متشکرم,HAPPY,0.0,سلام خیلی ممنون و متشکرم,"[سلام, ممنون, متشکرم]"
...,...,...,...,...,...
52105,یکی از بهترین ته چین هاییه که خوردم,HAPPY,0.0,یکی از بهترین ته چین هاییه که خوردم,"[ته, چین, هاییه, خوردم]"
52106,به موقع و خوب دستتون درد نکنه,HAPPY,0.0,به موقع و خوب دستتون درد نکنه,"[موقع, دستتون, درد, نکنه]"
52107,خیلی تازه و خوشمزه بود. بسته بندی شیک وعالی. خ...,HAPPY,0.0,خیلی تازه و خوشمزه بود بسته بندی شیک وعالی خدا...,"[تازه, خوشمزه, بسته, شیک, وعالی, خدابرکت, بده,..."
52108,فوق العاده سریع و عالی واقعا واسه همه زود آورد...,HAPPY,0.0,فوق‌العاده سریع و عالی واقعا واسه همه زود آورد...,"[فوق‌العاده, سریع, واقعا, واسه, زود, آوردنات, ..."


# train & test split

In [13]:
X_train = train_df["clean_text"]
y_train = train_df["label_id"]

X_test = test_df["clean_text"]
y_test = test_df["label_id"]

# TF-IDF

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from hazm.utils import stopwords_list

In [15]:
# create tfidf
stop_words = stopwords_list()
vectorize = TfidfVectorizer(
    max_features=15000,
    max_df=90,
    stop_words= stop_words,
    min_df= 5,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorize.fit_transform(X_train)
X_test_tfidf = vectorize.transform(X_test)

models = {
    "LogisticRegression" : LogisticRegression(),
    "LinearSVC" : LinearSVC(),
    "MultinomialNB" : MultinomialNB()
}

# train model
results = []
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    predict = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, predict)
    f1 = f1_score(y_test, predict)
    results.append({"approach": "TF-IDF", "model": name, "accuracy": acc, "f1": f1})
    print(f"=== {name} (TF-IDF) ===")
    print(classification_report(y_test, predict, target_names=['HAPPY','SAD']))

/home/milad/projects/env/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:412: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['آید', 'توان', 'تواند', 'توانند', 'رسد', 'رود', 'سال', 'نمی', 'های', 'گوید', 'گویند'] not in stop_words.
  warnings.warn(


=== LogisticRegression (TF-IDF) ===
              precision    recall  f1-score   support

       HAPPY       0.75      0.77      0.76      4486
         SAD       0.77      0.75      0.76      4547

    accuracy                           0.76      9033
   macro avg       0.76      0.76      0.76      9033
weighted avg       0.76      0.76      0.76      9033

=== LinearSVC (TF-IDF) ===
              precision    recall  f1-score   support

       HAPPY       0.74      0.77      0.76      4486
         SAD       0.76      0.73      0.75      4547

    accuracy                           0.75      9033
   macro avg       0.75      0.75      0.75      9033
weighted avg       0.75      0.75      0.75      9033

=== MultinomialNB (TF-IDF) ===
              precision    recall  f1-score   support

       HAPPY       0.77      0.76      0.77      4486
         SAD       0.77      0.78      0.77      4547

    accuracy                           0.77      9033
   macro avg       0.77      0.77 

# word2vec

In [16]:
from gensim.models import Word2Vec
import numpy as np

In [17]:
sentences = train_df["token"].to_list()
w2v_model = Word2Vec(
    sentences= sentences,
    vector_size= 100,
    window= 3,
    min_count= 5,
    epochs= 100,
    sg= 1
)

In [18]:
w2v_model.wv['خوشمزه']

array([ 0.36146244,  0.0981238 , -0.04876146, -0.2090513 ,  0.0534662 ,
       -0.3660064 , -0.39140046,  0.1771994 , -0.1362751 , -0.00481967,
       -0.24077757, -0.3476517 ,  0.18860792,  0.140763  ,  0.16988069,
       -0.13517998,  0.15604062, -0.44289666,  0.01440858, -0.26067287,
        0.2505115 ,  0.06560814, -0.19875465, -0.37846687, -0.02298697,
       -0.00641999, -0.32759693,  0.1831345 , -0.13791485,  0.3048164 ,
        0.38221753, -0.16747908,  0.08944332, -0.46723685, -0.19200295,
        0.31756854, -0.11434729,  0.03037162,  0.14030114, -0.02933482,
        0.0969149 , -0.08579095,  0.07817209,  0.07574669,  0.3830115 ,
        0.45187166, -0.10170163, -0.08670966,  0.2628743 , -0.03562219,
        0.2799386 , -0.20673512,  0.16172329,  0.11452065,  0.05519499,
       -0.07112488, -0.08458615,  0.2238138 ,  0.12612559,  0.4881189 ,
        0.33937898, -0.03469697,  0.01895845,  0.07171457,  0.04185758,
        0.11884557,  0.08161446,  0.32628182, -0.6061131 , -0.02

In [19]:
w2v_model.wv.most_similar('خوشمزه', topn=5)

[('خوش\u200cطعم', 0.7855303883552551),
 ('خوش\u200cمزه', 0.7042403817176819),
 ('لذیذ', 0.678156852722168),
 ('تازه', 0.6039243340492249),
 ('خوشمزه\u200cای', 0.5770860910415649)]

In [24]:
# create on sentence with words in sentence

def sentence_to_vec(tokens, model, dim= 100):
    vector = [model.wv[w] for w in tokens if w in model.wv]
    if len(vector) == 0:
        return np.zeros(dim)
    return np.mean(vector, axis=0)


X_train_emb = []
X_test_emb = []

for token in train_df["token"]:
    vec = sentence_to_vec(token, w2v_model)
    X_train_emb.append(vec)
X_train_emb = np.array(X_train_emb)


for token in test_df["token"]:
    vec = sentence_to_vec(token, w2v_model)
    X_test_emb.append(vec)
X_test_emb = np.array(X_test_emb)

In [25]:
X_train_emb[:5]

array([[-1.20767459e-01,  1.01883113e-01,  2.22383738e-02,
        -4.55788039e-02, -3.11101317e-01,  1.39642239e-01,
         4.39728528e-01,  5.40857553e-01, -3.02010894e-01,
        -1.94700837e-01,  1.35806456e-01, -2.95224875e-01,
        -3.52071285e-01,  3.33750904e-01,  1.60314441e-02,
        -1.36963978e-01, -7.30421990e-02,  3.61957967e-01,
        -2.63110638e-01, -2.92067230e-01,  1.16012186e-01,
         9.80601162e-02,  1.91727877e-01, -2.54610509e-01,
        -1.39365226e-01, -1.86918095e-01, -2.80898869e-01,
         1.73735678e-01, -1.77172065e-01,  2.02389210e-02,
        -1.18636370e-01, -1.16212498e-02,  1.96844757e-01,
        -2.87149735e-02, -3.89320493e-01,  1.81113631e-01,
         2.09853053e-01, -7.29210302e-02,  1.92312419e-01,
        -1.37829289e-01,  2.30653733e-01, -3.68802667e-01,
        -1.32564455e-01, -1.06288724e-01,  8.56952220e-02,
        -2.31557727e-01, -1.23963133e-01,  5.77983931e-02,
         3.27566326e-01,  2.43797719e-01,  2.59958953e-0

In [26]:
from sklearn.neural_network import MLPClassifier

In [27]:
model_emb = {
    "LogisticRegression" : LogisticRegression(max_iter= 1000),
    "LinearSVC" : LinearSVC(),
    "NLP_nn : " : MLPClassifier(hidden_layer_sizes=(100,), max_iter=500, random_state= 42)
}

emb_results = []
for name, model in model_emb.items():
    model.fit(X_train_emb, y_train)
    predict = model.predict(X_test_emb)
    acc = accuracy_score(y_test, predict)
    f1 = f1_score(y_test, predict)
    emb_results.append({"model": name, "accuracy": acc, "f1": f1})

    print(f"=== {name} ===")
    print(classification_report(y_test, predict, target_names=['HAPPY', 'SAD']))



=== LogisticRegression ===
              precision    recall  f1-score   support

       HAPPY       0.84      0.75      0.79      4486
         SAD       0.78      0.86      0.82      4547

    accuracy                           0.80      9033
   macro avg       0.81      0.80      0.80      9033
weighted avg       0.81      0.80      0.80      9033

=== LinearSVC ===
              precision    recall  f1-score   support

       HAPPY       0.85      0.74      0.79      4486
         SAD       0.77      0.87      0.82      4547

    accuracy                           0.81      9033
   macro avg       0.81      0.81      0.81      9033
weighted avg       0.81      0.81      0.81      9033

=== NLP_nn :  ===
              precision    recall  f1-score   support

       HAPPY       0.78      0.77      0.78      4486
         SAD       0.78      0.79      0.78      4547

    accuracy                           0.78      9033
   macro avg       0.78      0.78      0.78      9033
weighted av

/home/milad/projects/env/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


# save model

In [30]:
import joblib

joblib.dump(model_emb["LinearSVC"], "model/linear_svc.pkl")

w2v_model.save("model/word2vec.model")